In [2]:
import os
os.chdir("/home/ruslan/thesis/cl")

In [8]:
from trainers.models import RD4ADModel

model_rd4ad = RD4ADModel("cuda:0", 'resnet18', (224, 224))
model_rd4ad.load_model()
model = model_rd4ad.ad_model

In [9]:
import torch

model.load_state_dict(torch.load("/home/ruslan/thesis/tests/checkpoints/rd4ad_joint_training.pth", weights_only=True))

<All keys matched successfully>

In [10]:
model.eval()

ReverseDistillationModel(
  (encoder): TimmFeatureExtractor(
    (feature_extractor): FeatureListNet(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): BasicBlock(
          (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (drop_block): Identity()
          (act1): ReLU(inplace=True)
          (aa): Identity()
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (act2): ReLU(inplace=True)
        )
 

In [12]:
from datasets.full_dataset import CombinedDataset
from moviad.datasets.bmad.bmad_dataset import BMAD

data_full = CombinedDataset(BMAD, task_type="segmentation", root_dir="/mnt/disk1/ruslan_nuriev/bmad", norm=True)
data_test = data_full.load_test()
data_test = torch.utils.data.ConcatDataset(data_test)

(train) Task 0 (liver): 1542 samples
(train) Task 1 (chest): 8000 samples
(train) Task 2 (histopathology): 5088 samples
(train) Task 3 (brain): 7500 samples
(train) Task 4 (retinaoct): 26315 samples
(train) Task 5 (retinaresc): 4297 samples
(test) Task 0 (liver): 1493 samples
(test) Task 1 (chest): 17194 samples
(test) Task 2 (histopathology): 1997 samples
(test) Task 3 (brain): 3715 samples
(test) Task 4 (retinaoct): 968 samples
(test) Task 5 (retinaresc): 1805 samples


In [14]:
test_loader = torch.utils.data.DataLoader(data_test, batch_size=32, shuffle=False)

In [15]:
from moviad.utilities.evaluator import Evaluator

evaluator = Evaluator(test_loader, "cuda:0")
evaluator.evaluate(model)

Eval: 100%|██████████| 850/850 [07:51<00:00,  1.80it/s]


{'img_roc_auc': np.float64(0.5779009254371417),
 'pxl_roc_auc': np.float64(0.4798108873894098),
 'img_f1': np.float64(0.9202476703762369),
 'pxl_f1': np.float64(0.022687909237027938),
 'img_pr_auc': np.float64(0.8208010365247145),
 'pxl_pr_auc': np.float64(0.005799822664408517)}